# Build dữ liệu (chunk + lọc, câu nhân quả yếu) trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu từ Drive, cài thư viện) —
logic thật sự nằm trong repo (`src/chunking/`, `src/causal_detection/`,
`scripts/chunking/filter_chunks.py`, `scripts/causal_classification/build_causal_sentences.py`),
đồng bộ qua git. Xem `docs/data_pipeline_log.md` để biết chi tiết từng bước.

Notebook chạy 2 bước, đầu vào là `data/raw` (đã crawl sẵn ở local), đầu ra ghi vào
`data/chunks/` rồi `data/causal_sentences/` — đúng layout như chạy trên local:

1. `scripts.chunking.filter_chunks` — đọc `data/raw` → ghi `data/chunks/chunks_filtered.jsonl` + `chunks_rejected.jsonl`.
2. `scripts.causal_classification.build_causal_sentences` — đọc `data/chunks/chunks_filtered.jsonl` → ghi `data/causal_sentences/causal_sentences.csv`.

**Trước khi chạy:**
1. Trên Google Drive, tạo thư mục `CausalGraph/data/raw` rồi upload nguyên `data/raw`
   từ local lên đó (gồm `documents.jsonl` và các thư mục con như `web/`).
2. Đổi `DRIVE_ROOT` ở bước 3 bên dưới nếu bạn đặt thư mục Drive ở vị trí khác.
3. Không cần GPU cho bước này (embedding model chạy CPU vẫn ổn, chỉ chậm hơn) —
   có thể để Runtime mặc định, không cần đổi sang GPU.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [ ]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR

## 3. Gắn `data/` vào Drive

`data/` bị `.gitignore`, không nằm trong git — clone xong sẽ trống. Symlink sang Drive để
dùng đúng `data/raw` đã upload, và để `data/chunks`, `data/causal_sentences` sinh ra được
giữ lại trên Drive qua các session, không cần tải thủ công về máy sau khi chạy xong.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)

!rm -rf {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data

!ls "{DRIVE_ROOT}/data/raw" 2>/dev/null || echo "Chưa thấy data/raw trên Drive — upload trước khi chạy tiếp."

## 4. Cài thư viện

In [ ]:
!pip install -q -r requirements.txt

## 5. Chunk + lọc theo chủ đề

Đọc `data/raw`, chunk bằng `SemanticChunker`, chấm điểm liên quan chủ đề rồi ghi
`data/chunks/chunks_filtered.jsonl` (chunk giữ) và `chunks_rejected.jsonl` (chunk loại).
Chạy trên toàn bộ `data/raw` (~1.300 document) nên có thể mất một lúc — chạy lại được
an toàn nếu bị ngắt giữa chừng (ghi đè lại từ đầu, không resume dở dang).

In [ ]:
!python -m scripts.chunking.filter_chunks

## 6. Tách câu nhân quả yếu (weak label)

Đọc `data/chunks/chunks_filtered.jsonl`, tách câu, lọc câu <4 token, gán `weak_label`
bằng trigger, ghi `data/causal_sentences/causal_sentences.csv`.

⚠️ Nếu `data/causal_sentences/causal_sentences.csv` trên Drive đã có cột `human_label`
gán tay từ trước, cell này sẽ **ghi đè mất** (script mở file ở mode ghi mới, không merge).
Kiểm tra/backup trên Drive trước khi chạy nếu không chắc.

In [ ]:
!python -m scripts.causal_classification.build_causal_sentences

## 7. Kiểm tra nhanh kết quả

In [ ]:
!wc -l data/chunks/chunks_filtered.jsonl data/chunks/chunks_rejected.jsonl
!wc -l data/causal_sentences/causal_sentences.csv